# No Cow Left Behind: v40 Autopsy + Fix

**TL;DR**: Fork this, submit, get a competitive score. But if you want to learn *why* it works, keep reading.

I ran a post-mortem on [Ahmed Berat Ozer's v40](https://www.kaggle.com/code/ahmedberatozer/kaggriculture-v40-plans-that-fit-the-shops) — the strongest public notebook at the time — and found a **single recurring bug** costing **$24K–$64K per game**.

This notebook explains:
1. What the bug is (hint: dead cows)
2. How replay analysis revealed it
3. The 4 targeted patches that fix it
4. Arena results against 11 public agents

The fixed agent is pre-loaded from a companion dataset. Fork and submit, or read on and build your own improvements.

---

## The Crime Scene

After submitting v40 and watching the leaderboard replays, a pattern jumped out:

> **3 out of 4 losses had the exact same signature: 2 cows dying on day 1.**

Not day 10. Not day 29. **Day 1** — the cows starve because no worker is assigned to feed them in the opening turns.

Each dead cow costs **$12K–$32K** in lost milk revenue over the game. That's up to **$64K** evaporating before the farm even gets going.

v40's tape-replay architecture is excellent — 13 pre-optimized 720-turn action sequences covering different shop combinations. But certain shop-route combos have a gap: **no FEED action for cows in the first turns**. The tape just... forgets to feed them.

---

## The Investigation: Replay Forensics

I built a local arena ([the Colosseum](https://www.kaggle.com/code/nathanjacob/colosseum-876-2600-agent-testing-framework)) that runs games under LB-identical rules — same `kaggle_environments.make('kaggriculture')` call, same scoring, same RNG.

**Step 1: Download the competition.** I scraped 36+ public notebooks and extracted their agent code.

**Step 2: Run H2H tournaments.** v40 vs every downloaded agent, 20 games each (10 as player 0, 10 as player 1).

**Step 3: Find the pattern.** v40 crushes most agents (95–100% WR against the bottom half). But against the top 3, it loses consistently — and the replay logs always show the same thing: cow starvation on day 1.

The fix wasn't about a smarter strategy. It was about **finding and patching a specific failure mode** that the original optimization missed.

### The Methodology

This approach works on any public agent:
1. Download it
2. Run it in a local arena against strong opponents
3. Look at the losses — find the *recurring* pattern
4. Write a targeted reactive layer to handle that case
5. Test in the Colosseum: keep only if it wins more than it loses

No guesswork, no hyperparameter tuning — just forensic debugging.

---

## The 4 Fixes

Each fix was proposed, tested in a 30-game H2H against the unfixed version, and **only kept if it won more than it lost**.

### Fix 1: Labor Spawn Reassignment
When hired workers spawn at unexpected grid positions, v40's tape assumes they're somewhere they're not. The pathfinding breaks silently.

**Patch**: Permutation search over all possible worker→path assignments. Pick the assignment that minimizes total wasted moves.

**Test result**: 20W-0L-0T (100% WR, +$4,318/game)

### Fix 2: Animal Survival Guard (the big one)
Detects cows that will starve within the next day and **redirects the nearest wheat-carrying worker** to feed them — even if that worker was assigned to something else.

**Patch**: Every turn, check each animal's `unfed` counter. If unfed ≥ 2 (critical), dispatch any available worker. If unfed = 1 (at-risk), dispatch a non-HARVEST worker.

**Test result**: 29W-1L-0T (96.7% WR, +$3,048/game)

### Fix 3: Criticality Filter
Prevents the survival guard from pulling workers off active harvests for merely at-risk animals. Only truly critical (unfed=2) emergencies can interrupt a HARVEST.

**Test result**: 2W-0L-28T (+$129/game avg)

### Fix 4: Day-0 Preemptive Rescue
Extends the rescue window to turns 18–23 on day 0 — giving a 5-move window to feed animals *before* they hit the danger zone on day 1.

**Test result**: 2W-0L-28T (+$29/game avg)

> The first two fixes are the heavy hitters. Fixes 3–4 are refinements that add small but consistent edge.

---

## Arena Results

Full gauntlet: 20 games per opponent, LB-identical rules.

| Opponent | W-L-T | Win Rate | Avg Margin |
|----------|-------|----------|------------|
| **prvsiyan_soil** (frontier) | 0-20-0 | 0% | -$4,970 |
| **prvsiyan_moon** (frontier) | 1-19-0 | 5% | -$3,219 |
| hybrid_v1 | 8-12-0 | 40% | +$314 |
| pilkwang | 13-7-0 | 65% | +$3,534 |
| shop_router (yhay81 base) | 15-3-2 | 75% | +$98 |
| tetsu_shape | 19-1-0 | 95% | +$8,238 |
| yhay81_fieldbook | 20-0-0 | 100% | +$15,694 |
| kaito_v48 | 20-0-0 | 100% | +$15,919 |
| tschinkel_router | 20-0-0 | 100% | +$8,892 |
| boatlee_v16 | 20-0-0 | 100% | +$25,983 |
| tetsu_adaptive | 20-0-0 | 100% | +$29,000 |
| **Overall (220 games)** | **156-62-2** | **71%** | — |

Beats 9 out of 11 opponents. The two it can't beat (prvsiyan_soil / prvsiyan_moon) use a much more advanced **7-layer reactive stack** that I'll break down next.

---

## What's Still Beating Us (and How to Fix It)

The prvsiyan agents add reactive layers that v40 simply doesn't have:

| Layer | What It Does | Estimated Value |
|-------|-------------|----------------|
| Terminal fertilizer BFS | Collects extra fertilizer in last 7 turns | +$2–4K |
| SE quadrant monetization | Uses the idle 25% of the board for tomatoes/sheep | +$8–22K |
| Sell-before-buy ordering | SELLs execute first, prevents order truncation | +$1–3K |
| Cow-for-sheep substitution | Optimizes animal mix when no Yarn Store | +$3–5K |
| Wheat top-up | Prevents animal starvation cascades | +$2–4K |
| Day-1 wheat sale | Bridge cash for early hires | +$1–2K |

If you're looking to push this agent into the top 5%, these are your targets. Each one is a self-contained reactive layer — you can add them one at a time and test in the Colosseum.

### A Note on v41

Ahmed just released **v41** while I was writing this. The fixes in this notebook were developed independently through replay analysis of v40. The methodology — replay forensics → bug identification → targeted fix → deterministic testing — applies to **any version**. If v41 still has cow-death events, these patches apply directly. If v41 fixed them differently, the arena-testing pipeline here will tell you.

---

## Fork & Submit

The agent is loaded from the [companion dataset](https://www.kaggle.com/datasets/nathanjacob/kaggriculture-high-performance-agent). Just fork this notebook and hit **Submit to Competition**.

In [ ]:
import shutil
import tarfile
from pathlib import Path

WORK = Path('/kaggle/working')
INPUT = Path('/kaggle/input')

# --- Resilient file discovery (Kaggle mounts datasets inconsistently) ---
def find_file(name, search_root=INPUT):
    """Find a file by name under search_root, with rglob fallback."""
    # Try the expected direct path first
    direct = search_root / 'kaggriculture-high-performance-agent' / name
    if direct.exists():
        return direct
    # rglob fallback
    matches = list(search_root.rglob(name))
    if matches:
        return matches[0]
    return None

# Try submission.tar.gz first (pre-built, fastest)
src_tar = find_file('submission.tar.gz')
dst = WORK / 'submission.tar.gz'

if src_tar:
    shutil.copy2(src_tar, dst)
    print(f'Loaded agent: {dst} ({dst.stat().st_size / 1024:.1f} KB)')
else:
    # Fallback: build tar from main.py
    main_src = find_file('main.py')
    if main_src:
        main_dst = WORK / 'main.py'
        shutil.copy2(main_src, main_dst)
        with tarfile.open(dst, 'w:gz') as tar:
            tar.add(main_dst, arcname='main.py')
        main_dst.unlink()
        print(f'Built from main.py: {dst} ({dst.stat().st_size / 1024:.1f} KB)')
    else:
        # Fail-fast with directory listing for debugging
        print('ERROR: Agent not found. Contents of /kaggle/input:')
        for p in sorted(INPUT.rglob('*'))[:50]:
            print(f'  {p}')
        raise FileNotFoundError(
            'Neither submission.tar.gz nor main.py found under /kaggle/input. '
            'Make sure the dataset "kaggriculture-high-performance-agent" is attached.'
        )

In [ ]:
# Validate: make sure main.py is in the archive
with tarfile.open(dst, 'r:gz') as tar:
    members = tar.getnames()
    print(f'Archive contents: {members}')
    assert 'main.py' in members, 'main.py not found in archive!'

print(f'Size: {dst.stat().st_size / 1024:.1f} KB')
print('\nReady to submit. Go to Output tab -> Submit to Competition.')

---

## How the Automated Pipeline Works

The fixes above weren't hand-crafted by trial and error. They came from an **automated improvement loop**:

```
SCOUT  →  Download top 36+ public agents
  ↓
RANK   →  H2H tournament to find strongest agents
  ↓
ANALYZE → Replay forensics on losses (find the recurring bug)
  ↓
FIX    →  Write ONE targeted reactive layer
  ↓
JUDGE  →  30-game Colosseum test (LB-identical rules)
  ↓
DECIDE →  Keep if wins > losses, revert otherwise
```

Over 16 improvement attempts, **4 were kept and 12 were reverted**. The reverted ones include things like:
- `urgency_sort` — broke priority ordering (1W-3L, -$695/game)
- `day1_extended_window` — fired too early, disrupted planting (1W-29L, -$366/game)
- `final_window_critical` — diverted HARVEST workers incorrectly (1W-2L, -$163/game)

Most ideas that sound good actually hurt. **Deterministic testing catches this before you waste a submission slot.**

If you want the full testing framework, check out the [Colosseum notebook](https://www.kaggle.com/code/nathanjacob/colosseum-876-2600-agent-testing-framework).

---

*If this helped you, an upvote goes a long way. Happy farming!*